# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adarsh24BDA70227/flyrank-ml-track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will rank content items using two observable signals:

1. CTR opportunity: pages with at least 500 impressions, average position between 1 and 20, and CTR below 0.5%.
2. Search visibility: higher impressions receive a higher visibility score using a log-scaled transformation.

The final baseline score gives 60% weight to CTR opportunity and 40% weight to search visibility.

Reason codes:
- `low_ctr_high_visibility`
- `visibility_only`

Actions:
- `refresh_and_review_ctr`
- `monitor`

This is a transparent decision-support baseline. The score uses only observable current-window fields and does not use trend labels, future-window information, or product decision flags.

In [10]:
!git clone https://github.com/Adarsh24BDA70227/flyrank-ml-track.git

Cloning into 'flyrank-ml-track'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 141 (delta 52), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.86 MiB | 13.21 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [11]:
%cd flyrank-ml-track

/content/flyrank-ml-track/flyrank-ml-track


In [12]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [13]:
!ls data/raw

content_refresh_anonymized.csv


In [14]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

(30000, 44)


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from pathlib import Path

# Load Week-4 starter data
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# =========================================================
# AUDIT-ONLY LABEL
# =========================================================
# Used only to check whether our signals are directional.
# It will NOT be used in the baseline score.

df["audit_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# =========================================================
# SIGNAL 1 — CTR OPPORTUNITY
# =========================================================

df["low_ctr_visible"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

ctr_table = (
    df.groupby("low_ctr_visible")
      .agg(
          n=("audit_label", "size"),
          decline_rate=("audit_label", "mean")
      )
      .reset_index()
)

print("\nSIGNAL 1 — CTR OPPORTUNITY")
display(ctr_table)

ctr_true = ctr_table.loc[
    ctr_table["low_ctr_visible"] == True,
    "decline_rate"
].iloc[0]

ctr_false = ctr_table.loc[
    ctr_table["low_ctr_visible"] == False,
    "decline_rate"
].iloc[0]

if ctr_true > ctr_false:
    ctr_verdict = "CONFIRMED"
elif ctr_true < ctr_false:
    ctr_verdict = "OPPOSITE"
else:
    ctr_verdict = "MIXED"

print("Verdict:", ctr_verdict)


# =========================================================
# SIGNAL 2 — SEARCH VISIBILITY
# =========================================================

volume_bins = [0, 100, 500, 2000, 10000, np.inf]

volume_labels = [
    "1-100",
    "101-500",
    "501-2k",
    "2k-10k",
    "10k+"
]

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels,
    include_lowest=True
)

volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("audit_label", "size"),
          decline_rate=("audit_label", "mean")
      )
      .reset_index()
)

print("\nSIGNAL 2 — SEARCH VISIBILITY")
display(volume_table)

print(
    "\nVerdict: MIXED "
    "(higher-volume groups generally show higher decline rates, "
    "but the relationship is not perfectly monotonic)."
)



Dataset shape: (30000, 44)

SIGNAL 1 — CTR OPPORTUNITY


,low_ctr_visible,n,decline_rate
0,False,20241,0.501062
1,True,9759,0.627113


Verdict: CONFIRMED

SIGNAL 2 — SEARCH VISIBILITY


,volume_bucket,n,decline_rate
0,1-100,8006,0.389208
1,101-500,5279,0.604281
2,501-2k,6502,0.617964
3,2k-10k,6611,0.612918
4,10k+,3602,0.523598



Verdict: MIXED (higher-volume groups generally show higher decline rates, but the relationship is not perfectly monotonic).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# SECTION 2 — BUILD THE RANKED BASELINE QUEUE
# =========================================================

# ---------------------------------------------------------
# 1. CTR opportunity score
# ---------------------------------------------------------
# CTR is stored as a percentage in this dataset.
# Therefore 0.5 means 0.5%, not 50%.

ctr_gap = (
    (0.5 - df["ctr"]).clip(lower=0) / 0.5
)

# Only treat the CTR gap as an opportunity when the page
# has meaningful visibility and is on positions 1-20.
ctr_opportunity_score = (
    ctr_gap
    * (
        (df["impressions_90d"] >= 500)
        & (df["avg_position"] > 0)
        & (df["avg_position"] <= 20)
    ).astype(float)
)

df["ctr_opportunity_score"] = ctr_opportunity_score.clip(0, 1)


# ---------------------------------------------------------
# 2. Search visibility score
# ---------------------------------------------------------
# Log scaling prevents very large impression counts
# from dominating the ranking.

log_impressions = np.log1p(df["impressions_90d"])

visibility_min = log_impressions.min()
visibility_max = log_impressions.max()

df["visibility_score"] = (
    (log_impressions - visibility_min)
    / (visibility_max - visibility_min)
)


# ---------------------------------------------------------
# 3. ONE transparent baseline score
# ---------------------------------------------------------
# Fixed human-written weights.
# No model fitting is used.

df["baseline_action_score"] = (
    0.70 * df["ctr_opportunity_score"]
    + 0.30 * df["visibility_score"]
)


# ---------------------------------------------------------
# 4. ONE reason code per row
# ---------------------------------------------------------

df["reason_code"] = np.where(
    (
        (df["impressions_90d"] >= 500)
        & (df["avg_position"] > 0)
        & (df["avg_position"] <= 20)
        & (df["ctr"] < 0.5)
    ),
    "low_ctr_high_visibility",
    "visibility_only"
)


# ---------------------------------------------------------
# 5. ONE action label per row
# ---------------------------------------------------------

df["action"] = np.where(
    df["reason_code"] == "low_ctr_high_visibility",
    "review_ctr",
    "monitor"
)


# ---------------------------------------------------------
# 6. Rank the complete queue
# ---------------------------------------------------------

df["rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)


# ---------------------------------------------------------
# 7. Create ranked output
# ---------------------------------------------------------

output_columns = [
    "content_id",
    "client_id",
    "rank",
    "baseline_action_score",
    "reason_code",
    "action",
    "ctr_opportunity_score",
    "visibility_score",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue = (
    df[output_columns]
    .sort_values("rank")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Write required CSV
# ---------------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# 9. Precision@K + base rate
# ---------------------------------------------------------

def precision_at_k(scores, labels, k):
    order = np.argsort(
        -np.asarray(scores),
        kind="stable"
    )
    return np.asarray(labels)[order[:k]].mean()


base_rate = df["audit_label"].mean()

p10 = precision_at_k(
    df["baseline_action_score"],
    df["audit_label"],
    10
)

p20 = precision_at_k(
    df["baseline_action_score"],
    df["audit_label"],
    20
)

p50 = precision_at_k(
    df["baseline_action_score"],
    df["audit_label"],
    50
)

print("Base rate:", round(base_rate, 4))
print("Precision@10:", round(p10, 4))
print("Precision@20:", round(p20, 4))
print("Precision@50:", round(p50, 4))

print("\nCSV written to:")
print(output_path)

print("\nTop 20 ranked rows:")
display(queue.head(20))


Base rate: 0.5421
Precision@10: 0.5
Precision@20: 0.7
Precision@50: 0.62

CSV written to:
work/outputs/baseline_action_score.csv

Top 20 ranked rows:


,content_id,client_id,rank,baseline_action_score,reason_code,action,ctr_opportunity_score,visibility_score,impressions_90d,avg_position,ctr
0,content_c8e9d6ab9013,client_19581e27de,1,0.978130,low_ctr_high_visibility,review_ctr,1.00,0.927100,208678,9.7,0.00
1,content_453722754fea,client_f369cb89fc,2,0.954536,low_ctr_high_visibility,review_ctr,0.98,0.895121,140079,7.6,0.01
2,content_4a6607efcb46,client_6208ef0f77,3,0.952379,low_ctr_high_visibility,review_ctr,0.98,0.887929,128068,2.2,0.01
3,content_39881853ef0c,client_f369cb89fc,4,0.949245,low_ctr_high_visibility,review_ctr,0.98,0.877483,112434,7.2,0.01
4,content_8451fc6f034d,client_d029fa3a95,5,0.942521,low_ctr_high_visibility,review_ctr,0.94,0.948404,272144,2.3,0.03
5,content_c84a0ab98e90,client_f369cb89fc,6,0.937757,low_ctr_high_visibility,review_ctr,0.94,0.932523,223271,7.8,0.03
6,content_0919dd345d80,client_4e07408562,7,0.936655,low_ctr_high_visibility,review_ctr,0.96,0.882183,119217,7.0,0.02
7,content_d274ac4158ef,client_4e07408562,8,0.936107,low_ctr_high_visibility,review_ctr,0.98,0.833689,65138,6.8,0.01
8,content_e5f459e737b7,client_f369cb89fc,9,0.932624,low_ctr_high_visibility,review_ctr,0.98,0.822080,56363,5.9,0.01
9,content_339b357d04c7,client_bbb965ab0c,10,0.928190,low_ctr_high_visibility,review_ctr,0.98,0.807299,46879,3.7,0.01


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

I reviewed the highest-ranked 20 items from the baseline queue.

For each item, I record:
- the recommended action,
- the reason code that caused the ranking,
- a confidence note based on the observed signals,
- and a condition that could make the recommendation wrong.

These are decision-support observations, not guaranteed outcomes.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# SECTION 3 — TOP-20 REVIEW
# =========================================================

top20 = queue.head(20).copy()


def make_confidence_note(row):
    if row["reason_code"] == "low_ctr_high_visibility":
        return (
            "Higher confidence: the item satisfies the visible "
            "low-CTR opportunity condition."
        )
    else:
        return (
            "Lower confidence: ranking is mainly supported by "
            "search visibility rather than the CTR opportunity signal."
        )


def make_wrong_condition(row):
    if row["reason_code"] == "low_ctr_high_visibility":
        return (
            "Could be wrong if the low CTR is intentional, "
            "the CTR measurement is unreliable, or the page does "
            "not have a genuine CTR improvement opportunity."
        )
    else:
        return (
            "Could be wrong if high impressions do not represent "
            "a useful actionable opportunity."
        )


top20["confidence_note"] = top20.apply(
    make_confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    make_wrong_condition,
    axis=1
)


review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_action_score",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns].copy()

print("Top-20 review:")
display(top20_review)


Top-20 review:


,rank,content_id,action,reason_code,baseline_action_score,confidence_note,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,review_ctr,low_ctr_high_visibility,0.978130,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
1,2,content_453722754fea,review_ctr,low_ctr_high_visibility,0.954536,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
2,3,content_4a6607efcb46,review_ctr,low_ctr_high_visibility,0.952379,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
3,4,content_39881853ef0c,review_ctr,low_ctr_high_visibility,0.949245,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
4,5,content_8451fc6f034d,review_ctr,low_ctr_high_visibility,0.942521,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
5,6,content_c84a0ab98e90,review_ctr,low_ctr_high_visibility,0.937757,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
6,7,content_0919dd345d80,review_ctr,low_ctr_high_visibility,0.936655,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
7,8,content_d274ac4158ef,review_ctr,low_ctr_high_visibility,0.936107,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
8,9,content_e5f459e737b7,review_ctr,low_ctr_high_visibility,0.932624,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."
9,10,content_339b357d04c7,review_ctr,low_ctr_high_visibility,0.928190,Higher confidence: the item satisfies the visi...,"Could be wrong if the low CTR is intentional, ..."



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

The baseline is a decision-support ranking, so some high-ranked items can still be weak picks.

I reviewed high-ranked items that rely mainly on visibility rather than the stronger CTR opportunity signal.

I also checked the scoring inputs for leakage. The baseline score does not use trend labels, trend percentage, future-window fields, or product decision flags.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# =========================================================

# ---------------------------------------------------------
# 1. Identify weak-pick candidates
# ---------------------------------------------------------

weak_picks = queue[
    queue["reason_code"] == "visibility_only"
].head(5).copy()

print("Weak-pick candidates:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_action_score",
            "reason_code",
            "action",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ]
)


# ---------------------------------------------------------
# 2. Explain why these can be weak
# ---------------------------------------------------------

print("\nWhy these may be weak:")
print(
    "These items receive ranking support from visibility, "
    "but they do not satisfy the explicit low-CTR opportunity "
    "condition. High impressions alone do not prove that an "
    "action is needed."
)


# ---------------------------------------------------------
# 3. Leakage check
# ---------------------------------------------------------

score_features = [
    "ctr_opportunity_score",
    "visibility_score"
]

print("\nScoring features used:")
print(score_features)

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nForbidden / label-derived feature check:")

for feature in forbidden_features:
    print(
        feature,
        "→ NOT USED IN SCORE"
    )


# ---------------------------------------------------------
# 4. Explicit product-flag check
# ---------------------------------------------------------

product_flag_candidates = [
    column
    for column in df.columns
    if "flag" in column.lower()
]

print("\nColumns containing 'flag':")
print(product_flag_candidates)

print(
    "\nProduct decision flags are not used as scoring features."
)

print(
    "Future-window fields are not used as scoring features."
)


Weak-pick candidates:


,rank,content_id,baseline_action_score,reason_code,action,impressions_90d,avg_position,ctr
9096,9097,content_2cb567c3c89b,0.299052,visibility_only,monitor,497727,22.2,0.10
9114,9115,content_2dba2b1f9536,0.296272,visibility_only,monitor,443434,27.9,0.21
9157,9158,content_2c2606c5d176,0.290398,visibility_only,monitor,347399,4.2,0.53
9179,9180,content_44e481c8f55b,0.287864,visibility_only,monitor,312694,1.4,0.65
9184,9185,content_9532f197bbc8,0.287593,visibility_only,monitor,309192,2.0,0.87



Why these may be weak:
These items receive ranking support from visibility, but they do not satisfy the explicit low-CTR opportunity condition. High impressions alone do not prove that an action is needed.

Scoring features used:
['ctr_opportunity_score', 'visibility_score']

Forbidden / label-derived feature check:
trend_direction → NOT USED IN SCORE
trend_pct → NOT USED IN SCORE
is_declining_label → NOT USED IN SCORE

Columns containing 'flag':
[]

Product decision flags are not used as scoring features.
Future-window fields are not used as scoring features.


## Self-check

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] Two signals were checked with visible bucket tables and n
- [x] CTR opportunity signal verdict is CONFIRMED
- [x] Search visibility signal verdict is MIXED
- [x] At least one signal is linked to FlyRank's CTR-fix logic
- [x] One transparent baseline score is used
- [x] Each ranked row has one reason code
- [x] Each ranked row has one action label
- [x] Ranked queue is written to `work/outputs/baseline_action_score.csv`
- [x] Precision@K and base rate are reported
- [x] Top 20 rows are reviewed
- [x] Weak picks are discussed
- [x] Trend labels are not used as scoring features
- [x] No future-window information is used
- [x] Product decision flags are not used as scoring features
- [x] Claims use careful decision-support language
- [x] No client names, private queries, or unnecessary URLs are included
- [x] Notebook is ready to be committed under `work/notebooks/`